# 06 — Stacking với AdaBoost meta-learner

Nạp bộ ba từ notebook 05 → tạo xác suất OOF 10-fold → fit AdaBoost → so sánh với mô hình đơn và soft voting. Code là trọng tâm; lý thuyết chi tiết trình bày trong report.

**Trước khi chạy:** hoàn thành notebook 05 để có `selected_estimators.joblib` và `split_indices.npz`. Đặt notebook trong `notebooks/` hoặc chạy từ thư mục gốc đồ án. Kết quả lưu tại `results/stacking_adaboost/`.

Đây là stacking, không phải AdaBoost trực tiếp trên 784 pixel. Ba base model cho 30 đặc trưng xác suất, theo thứ tự mô hình và lớp được lưu trong file phân lớp.


## 1. Import và cấu hình

Giữ seed 42 và 10 fold. Mặc định tắt cache để thời gian chạy phản ánh một lượt huấn luyện mới; có thể bật `USE_CACHE=True` khi cần tái sử dụng các fold. Không thay bộ ba thủ công sau khi xem test.


In [ ]:
from pathlib import Path
import sys, time, json, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from joblib import Memory, dump, load
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import load_prepared_mnist

SEED, N_FOLDS, USE_CACHE = 42, 10, False
CLASSES = np.arange(10)
OUTPUT = ROOT / 'results' / 'stacking_adaboost'
OUTPUT.mkdir(parents=True, exist_ok=True)
memory = Memory(OUTPUT / 'cache' if USE_CACHE else None, verbose=0)
selection_dir = ROOT / 'results' / 'select_base_models'


## 2. Nạp đúng bộ ba và split từ notebook 05

Chỉ số split tham chiếu 60.000 ảnh train gốc. Pixel vẫn chia 255 và làm phẳng bằng `data.py`. Test giữ riêng, không tham gia tạo OOF hoặc chọn meta-parameters.


In [ ]:
base_models = load(selection_dir / 'selected_estimators.joblib')
indices = np.load(selection_dir / 'split_indices.npz')
X, y, X_test, y_test = load_prepared_mnist()
X_train, y_train = X[indices['train_idx']], y[indices['train_idx']]
X_val, y_val = X[indices['val_idx']], y[indices['val_idx']]
del X
names = list(base_models)
assert len(names) == 3 and not np.intersect1d(indices['train_idx'], indices['val_idx']).size
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)
display(pd.DataFrame({'model': names, 'config': [str(base_models[name]) for name in names]}))


## 3. Tạo đặc trưng OOF

Mỗi ảnh nhận xác suất từ một base model fit trên 9 fold còn lại. Cấu hình được giữ cố định từ notebook 05; đây không phải nested cross-validation vì bước tuning/chọn mô hình đã thực hiện trước đó.

Meta-model chỉ fit trên `Z_train` OOF. Không dùng xác suất in-sample của base model để thay thế. Kiểm tra mỗi ảnh được gán đúng một holdout fold.


In [ ]:
@memory.cache
def fold_proba(estimator, X_fit, y_fit, X_holdout):
    start = time.perf_counter()
    model = clone(estimator).fit(X_fit, y_fit)
    fit_s = time.perf_counter() - start
    proba = model.predict_proba(X_holdout)[:, np.argsort(model.classes_)]
    return proba, fit_s, time.perf_counter() - start

folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_train, y_train))
seen, fold_id = np.zeros(len(y_train), dtype=int), np.zeros(len(y_train), dtype=int)
for k, (_, hold) in enumerate(folds):
    seen[hold] += 1
    fold_id[hold] = k
assert np.all(seen == 1)
Z_train = np.zeros((len(y_train), len(CLASSES) * len(names)), dtype=np.float32)
oof_rows = []
start = time.perf_counter()
for j, name in enumerate(names):
    for k, (fit, hold) in enumerate(folds, 1):
        proba, fit_s, fold_s = fold_proba(base_models[name], X_train[fit], y_train[fit], X_train[hold])
        Z_train[hold, 10*j:10*(j+1)] = proba
        oof_rows.append(dict(model=name, fold=k, recorded_fit_seconds=fit_s, recorded_fold_seconds=fold_s))
        print(f'{name}: fold {k}/{N_FOLDS}', flush=True)
oof_wall_s = time.perf_counter() - start
oof_times = pd.DataFrame(oof_rows)
print('OOF:', Z_train.shape)


## 4. Refit base models và tạo đặc trưng validation

Fit từng base model trên toàn bộ 50.000 train rồi dự đoán validation. Luôn sắp xác suất theo lớp 0–9. Dự đoán model đơn dùng `predict`; soft voting và stacking dùng `predict_proba` (hai quyết định có thể khác nhau với SVM).


In [ ]:
@memory.cache
def fit_base(estimator, X, y):
    start = time.perf_counter()
    model = clone(estimator).fit(X, y)
    return model, time.perf_counter() - start

def predict_bases(X):
    probabilities, predictions, rows = {}, {}, []
    for name in names:
        model = fitted[name]
        start = time.perf_counter()
        probabilities[name] = model.predict_proba(X)[:, np.argsort(model.classes_)]
        proba_s = time.perf_counter() - start
        start = time.perf_counter()
        predictions[name] = model.predict(X)
        rows.append(dict(model=name, proba_seconds=proba_s, predict_seconds=time.perf_counter()-start))
    return probabilities, predictions, rows

fitted, base_rows = {}, []
start = time.perf_counter()
for name in names:
    fitted[name], fit_s = fit_base(base_models[name], X_train, y_train)
    base_rows.append(dict(model=name, recorded_fit_seconds=fit_s))
refit_wall_s = time.perf_counter() - start
base_times = pd.DataFrame(base_rows)
val_proba, val_pred, val_times = predict_bases(X_val)
Z_val = np.concatenate([val_proba[name] for name in names], axis=1)
voting_acc = accuracy_score(y_val, np.mean([val_proba[name] for name in names], axis=0).argmax(1))


## 5. Khảo sát AdaBoost meta-learner

Khảo sát từng tham số quanh baseline `learning_rate=0.5, max_depth=2, n_estimators=100`, không khảo sát mọi tổ hợp. Tái sử dụng baseline cho khảo sát độ sâu. Số cây được lấy bằng `staged_predict` từ một mô hình tối đa 300 cây, tránh fit riêng từng số cây. AdaBoost có thể dừng sớm; chỉ lưu các stage thực tế.

Các biểu đồ có đường accuracy soft voting để đối chiếu. Meta-model luôn fit trên OOF, chọn tham số bằng validation. Dùng SAMME; nhánh `algorithm` nhỏ chỉ để tương thích các phiên bản sklearn còn tham số này.


In [ ]:
def make_meta(lr=0.5, depth=2, trees=100):
    params = dict(estimator=DecisionTreeClassifier(max_depth=depth, random_state=SEED),
                  n_estimators=trees, learning_rate=lr, random_state=SEED)
    if 'algorithm' in inspect.signature(AdaBoostClassifier).parameters:
        params['algorithm'] = 'SAMME'
    return AdaBoostClassifier(**params)

def score_row(pred, lr, depth, trees):
    _, _, f1, _ = precision_recall_fscore_support(y_val, pred, average='macro', zero_division=0)
    return dict(learning_rate=lr, max_depth=depth, n_estimators=trees,
                accuracy=accuracy_score(y_val, pred), f1_macro=f1)

def show_results(table, parameter):
    display(table.round(5))
    ax = table.plot(x=parameter, y=['accuracy', 'f1_macro'], marker='o', figsize=(7, 4))
    ax.axhline(voting_acc, color='gray', linestyle='--', label='Soft voting accuracy')
    ax.set(ylabel='Điểm validation', title=f'AdaBoost — {parameter}')
    ax.legend(); ax.grid(alpha=.2)
    ax.figure.tight_layout(); ax.figure.savefig(OUTPUT / f'survey_{parameter}.png', dpi=300)
    plt.show()

meta_search_fit_s = 0.0


### 5.1. Learning rate


In [ ]:
lr_rows = []
for lr in [0.01, 0.1, 0.2, 0.5, 1.0]:
    model = make_meta(lr=lr)
    start = time.perf_counter(); model.fit(Z_train, y_train)
    meta_search_fit_s += time.perf_counter() - start
    lr_rows.append(score_row(model.predict(Z_val), lr, 2, 100))
lr_results = pd.DataFrame(lr_rows)
show_results(lr_results, 'learning_rate')


### 5.2. Độ sâu cây


In [ ]:
depth_rows = []
for depth in [1, 2, 3, 4]:
    if depth == 2:
        row = lr_results.loc[lr_results.learning_rate == 0.5].iloc[0].to_dict()
    else:
        model = make_meta(depth=depth)
        start = time.perf_counter(); model.fit(Z_train, y_train)
        meta_search_fit_s += time.perf_counter() - start
        row = score_row(model.predict(Z_val), 0.5, depth, 100)
    depth_rows.append(row)
depth_results = pd.DataFrame(depth_rows)
show_results(depth_results, 'max_depth')


### 5.3. Số cây


In [ ]:
tree_counts = [1, 20, 50, 100, 200, 300]
model = make_meta(trees=300)
start = time.perf_counter(); model.fit(Z_train, y_train)
meta_search_fit_s += time.perf_counter() - start
tree_rows = [score_row(pred, 0.5, 2, trees) for trees, pred in enumerate(model.staged_predict(Z_val), 1) if trees in tree_counts]
tree_results = pd.DataFrame(tree_rows)
show_results(tree_results, 'n_estimators')


### 5.4. Chốt cấu hình

Chọn accuracy validation cao nhất, sau đó F1 macro; nếu vẫn hòa ưu tiên cây nông, ít cây và learning rate nhỏ hơn. Fit lại cấu hình được chọn trên OOF để đo riêng thời gian fit meta-model cuối. Không sử dụng test để chọn.


In [ ]:
survey = pd.concat([lr_results, depth_results, tree_results], ignore_index=True).drop_duplicates(['learning_rate', 'max_depth', 'n_estimators'])
ranked = survey.sort_values(['accuracy', 'f1_macro', 'max_depth', 'n_estimators', 'learning_rate'], ascending=[False, False, True, True, True])
best = ranked.iloc[0]
meta = make_meta(float(best.learning_rate), int(best.max_depth), int(best.n_estimators))
start = time.perf_counter(); meta.fit(Z_train, y_train)
meta_fit_s = time.perf_counter() - start
display(ranked.round(5))
print('Cấu hình cuối:', meta)


**Nhận xét:** so sánh điểm stacking với soft voting và model đơn; không mặc định stacking phải thắng. Cùng validation đã dùng chọn bộ ba và meta-parameters nên điểm validation có thể lạc quan; đây không phải ước lượng nested CV.


## 6. Đánh giá train / validation / test và thời gian dự đoán

Train bên dưới là **in-sample của pipeline sau refit**, không phải điểm OOF tổng quát hóa. Điểm `meta.predict(Z_train)` cũng không phải đánh giá độc lập vì meta-model đã học trên chính `Z_train`. Test chỉ đánh giá sau khi chốt cấu hình.

Đo riêng xác suất base models, dự đoán nhãn model đơn và bước gộp. `predict_seconds` của stacking/voting = tổng thời gian tạo xác suất + bước gộp; không cộng thêm thời gian dự đoán nhãn model đơn. Các thời gian này dùng chung một lượt tính xác suất, không phải benchmark độc lập.


In [ ]:
def evaluate(y, predictions, split):
    rows = []
    for name, pred in predictions.items():
        precision, recall, f1, _ = precision_recall_fscore_support(y, pred, labels=CLASSES, average='macro', zero_division=0)
        rows.append(dict(model=name, split=split, accuracy=accuracy_score(y, pred),
                         precision_macro=precision, recall_macro=recall, f1_macro=f1))
    return pd.DataFrame(rows)

recorded_stack_fit_s = oof_times.recorded_fold_seconds.sum() + base_times.recorded_fit_seconds.sum() + meta_fit_s
all_results, prediction_times, saved_predictions = [], [], {}
for split, X_split, y_split in [('Train', X_train, y_train), ('Validation', X_val, y_val), ('Test', X_test, y_test)]:
    if split == 'Validation':
        probabilities, predictions, timing_rows = val_proba, dict(val_pred), val_times
    else:
        probabilities, predictions, timing_rows = predict_bases(X_split)
    proba_s = sum(row['proba_seconds'] for row in timing_rows)
    start = time.perf_counter()
    Z = np.concatenate([probabilities[name] for name in names], axis=1)
    predictions['Stacking AdaBoost'] = meta.predict(Z)
    stacking_extra_s = time.perf_counter() - start
    start = time.perf_counter()
    predictions['Soft voting'] = np.mean([probabilities[name] for name in names], axis=0).argmax(1)
    voting_extra_s = time.perf_counter() - start
    table = evaluate(y_split, predictions, split)
    fit_times = dict(zip(base_times.model, base_times.recorded_fit_seconds))
    fit_times.update({'Soft voting': base_times.recorded_fit_seconds.sum(), 'Stacking AdaBoost': recorded_stack_fit_s})
    predict_times = {row['model']: row['predict_seconds'] for row in timing_rows}
    predict_times.update({'Soft voting': proba_s+voting_extra_s, 'Stacking AdaBoost': proba_s+stacking_extra_s})
    table['fit_seconds'] = table.model.map(fit_times)
    table['predict_seconds'] = table.model.map(predict_times)
    all_results.append(table); saved_predictions[split] = predictions
    prediction_times.extend([dict(split=split, **row) for row in timing_rows])
results = pd.concat(all_results, ignore_index=True)
display(results.round(5))
test_results = results[results.split == 'Test']
stack_acc = float(test_results.loc[test_results.model == 'Stacking AdaBoost', 'accuracy'].iloc[0])
best_single_acc = test_results[test_results.model.isin(names)].accuracy.max()
print(f'Stacking test accuracy={stack_acc:.2%}; chênh model đơn tốt nhất: {(stack_acc-best_single_acc)*100:+.2f} điểm phần trăm')


## 7. Trực quan kết quả và phân tích lỗi

Vẽ confusion matrix stacking trên ba tập và bảng kết quả test có cả voting và base models. Không thay mô hình sau khi xem các kết quả này.


In [ ]:
datasets = {'Train': y_train, 'Validation': y_val, 'Test': y_test}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (split, target) in zip(axes, datasets.items()):
    ConfusionMatrixDisplay.from_predictions(target, saved_predictions[split]['Stacking AdaBoost'], labels=CLASSES, ax=ax,
        cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(f'Stacking AdaBoost — {split}')
fig.tight_layout(); fig.savefig(OUTPUT / 'confusion_matrices.png', dpi=300); plt.show()
ax = test_results.set_index('model')[['accuracy', 'f1_macro']].plot.bar(figsize=(9, 4), ylim=(0, 1))
ax.set(xlabel='', ylabel='Điểm số', title='So sánh trên test')
plt.xticks(rotation=20, ha='right'); ax.figure.tight_layout()
ax.figure.savefig(OUTPUT / 'comparison_test.png', dpi=300); plt.show()
report = pd.DataFrame(classification_report(y_test, saved_predictions['Test']['Stacking AdaBoost'], output_dict=True, zero_division=0)).T
display(report.loc[[str(k) for k in CLASSES]].round(4))


## 8. Lưu kết quả và bộ phân lớp

`fit_seconds` của stacking gồm tổng fit và tạo xác suất OOF trong các fold + refit base models + fit meta cuối, **chưa gồm tìm kiếm tham số**. Khi bật cache, đây là thời gian fit đã ghi từ lần tính đầu, không phải thời gian chạy lại hiện tại. Lưu riêng wall time và cờ cache; không cộng lặp cùng thời gian fit qua ba split.

File phân lớp giữ base models đã fit, meta-model, thứ tự mô hình và lớp. Suy luận: nạp → lấy xác suất theo đúng thứ tự → ghép cột → `meta.predict`. Chỉ nạp joblib do chính bạn tạo. Chạy lại ghi đè file cùng tên.


In [ ]:
results.to_csv(OUTPUT / 'metrics.csv', index=False)
results.query("model == 'Stacking AdaBoost' and split == 'Test'").to_csv(OUTPUT / 'comparison.csv', index=False)
survey.to_csv(OUTPUT / 'hyperparameters.csv', index=False)
ranked.head(1).to_csv(OUTPUT / 'selected_config.csv', index=False)
oof_times.to_csv(OUTPUT / 'oof_times.csv', index=False)
base_times.to_csv(OUTPUT / 'base_times.csv', index=False)
pd.DataFrame(prediction_times).to_csv(OUTPUT / 'prediction_times.csv', index=False)
report.to_csv(OUTPUT / 'classification_report_test.csv')
for split, target in datasets.items():
    for name, pred in saved_predictions[split].items():
        cm = confusion_matrix(target, pred, labels=CLASSES)
        pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUTPUT / f'{name.replace(" ", "_")}_confusion_{split.lower()}.csv', index_label='true_digit')
np.savez_compressed(OUTPUT / 'meta_features.npz', Z_train=Z_train, y_train=y_train, fold_id=fold_id)
config = dict(seed=SEED, folds=N_FOLDS, model_order=names, classes=CLASSES.tolist(), cache_enabled=USE_CACHE,
    learning_rate=float(best.learning_rate), max_depth=int(best.max_depth), n_estimators=int(best.n_estimators),
    actual_estimators=len(meta.estimators_), oof_wall_seconds=oof_wall_s, refit_wall_seconds=refit_wall_s,
    pipeline_training_wall_seconds=oof_wall_s + refit_wall_s + meta_fit_s,
    recorded_stack_fit_seconds=float(recorded_stack_fit_s), meta_search_fit_seconds=meta_search_fit_s,
    meta_final_fit_seconds=meta_fit_s, sklearn_version=sklearn.__version__)
(OUTPUT / 'config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
dump(dict(base_models=fitted, meta_model=meta, model_order=names, classes=CLASSES, seed=SEED, folds=N_FOLDS), OUTPUT / 'classifier.joblib')
print('Đã lưu:', OUTPUT)
